# TTS Model Comparison: Chatterbox, Seamless M4T v2, and VibeVoice

This notebook provides a comprehensive comparison of three text-to-speech (TTS) models for potential use in production applications.

## Quick Summary

| Feature | Chatterbox | Seamless M4T v2 | VibeVoice Realtime |
|---------|------------|-----------------|--------------------|
| **License** | MIT | CC-BY-NC 4.0 | Research Only |
| **Commercial Use** | Yes | No | No |
| **Sample Rate** | 24kHz | 16kHz | 24kHz |
| **Streaming** | Excellent | No | Excellent (~300ms) |
| **Multi-speaker** | No (single) | Yes (8 speakers) | No (Carter only) |
| **Multilingual** | English only | 100+ languages | English only |
| **Memory** | ~8GB | ~8GB | ~4GB |
| **Quality** | High | Moderate (robotic) | High |

---
# 1. Chatterbox TTS

**Developer:** Resemble AI  
**Model:** chatterbox-tts  
**Architecture:** Proprietary neural TTS

## 1.1 License

- **License:** MIT License
- **Commercial Use:** Yes - Enterprise friendly
- **Watermarking:** All output includes an imperceptible audio watermark

> **Recommendation:** Best choice for commercial/enterprise applications due to permissive MIT license.

## 1.2 Installation

```bash
# For Python 3.11 (recommended)
pip install chatterbox-tts

# For Python 3.12+ (requires workaround)
pip install numpy>=1.26.0
pip install chatterbox-tts --no-deps
pip install conformer==0.3.2 diffusers==0.29.0 gradio librosa==0.11.0 omegaconf pykakasi==2.3.0 pyloudnorm resemble-perth==1.0.1 s3tokenizer safetensors spacy-pkuseg torchaudio transformers==4.46.3
```

## 1.3 Tunable Parameters

| Parameter | Range | Default | Description |
|-----------|-------|---------|-------------|
| `exaggeration` | 0.0 - 1.0 | 0.5 | Expressiveness level (higher = more expressive) |
| `cfg_weight` | 0.0 - 1.0 | 0.5 | Classifier-free guidance weight |
| `temperature` | 0.1 - 1.5 | 0.8 | Sampling randomness |
| `top_p` | 0.5 - 1.0 | 0.95 | Nucleus sampling threshold |

### Recommended Settings

**Professional/Formal Voice:**
```python
exaggeration=0.3, cfg_weight=0.5, temperature=0.7
```

**Expressive/Casual Voice:**
```python
exaggeration=0.7, temperature=0.9
```

## 1.4 Special Features

### Paralinguistic Tags
Chatterbox uniquely supports emotional expressions via tags:

```python
# Supported tags
[laugh]    # Laughter
[chuckle]  # Light chuckle
[sigh]     # Sighing
[gasp]     # Gasping
[cough]    # Coughing

# Example usage
text = "[chuckle] That's a great answer! I really appreciate your enthusiasm."
```

### Platform Support
- CUDA (NVIDIA) - Best performance
- MPS (Apple Silicon) - Native support, works great
- DirectML (AMD on Windows) - Supported
- CPU - Fallback option

## 1.5 Known Issues

- **Watermarking:** All audio contains imperceptible watermark (cannot be disabled)
- **English Only:** Turbo model is English-only; multilingual requires separate model
- **Memory:** Requires ~8GB for comfortable inference
- **Python 3.12:** Has dependency issues with numpy pinning (use workaround above)

In [ ]:
# Chatterbox Basic Usage Example
from chatterbox import ChatterboxTTS

# Load model
model = ChatterboxTTS.from_pretrained(device="cuda")  # or "mps", "cpu"

# Generate with recommended settings
text = "Hello, thank you for applying to this position."
wav = model.generate(
    text,
    exaggeration=0.3,
    cfg_weight=0.5,
    temperature=0.7
)

# Save audio
import soundfile as sf
sf.write("output.wav", wav.cpu().numpy().squeeze(), model.sr)

## 1.6 Audio Samples - Chatterbox

*Add your audio files here*

In [ ]:
# Placeholder for Chatterbox audio samples
from IPython.display import Audio, display

# Uncomment and update paths to your audio files
# display(Audio("outputs/chatterbox/recommended_phrase_1.wav"))
# display(Audio("outputs/chatterbox/recommended_phrase_2.wav"))

---
# 2. Seamless M4T v2

**Developer:** Meta (Facebook AI)  
**Model:** facebook/seamless-m4t-v2-large  
**Architecture:** Transformer-based multimodal model

## 2.1 License

- **License:** CC-BY-NC 4.0 (Creative Commons Attribution-NonCommercial)
- **Commercial Use:** No - Non-commercial only

> **Warning:** Cannot be used for commercial applications without a separate license agreement with Meta.

## 2.2 Installation

```bash
pip install transformers torch torchaudio sentencepiece
```

The model will be downloaded automatically from Hugging Face on first use (~10GB).

## 2.3 Tunable Parameters

| Parameter | Range | Default | Description |
|-----------|-------|---------|-------------|
| `num_beams` | 1 - 10 | 5 | Beam search width (higher = more stable but slower) |
| `do_sample` | True/False | False | Enable sampling for variety |
| `temperature` | 0.1 - 2.0 | 1.0 | Expressiveness (requires do_sample=True) |
| `speaker_id` | 0 - 7 | 0 | Different voice embeddings |
| `src_lang` | string | "eng" | Source language code |
| `tgt_lang` | string | "eng" | Target language code |

### Recommended Settings (Less Robotic)
```python
speaker_id=1, do_sample=True, temperature=0.7, num_beams=1
```

## 2.4 Special Features

### Multilingual Support
Seamless supports 100+ languages for both input and output:

```python
# English to Spanish TTS
output = model.generate(
    **inputs,
    tgt_lang="spa"  # Spanish output
)

# French input, German output
inputs = processor(text=french_text, src_lang="fra", return_tensors="pt")
output = model.generate(**inputs, tgt_lang="deu")
```

### Multi-Speaker
8 different speaker embeddings available (speaker_id 0-7)

## 2.5 Known Issues

- **Robotic Sound:** The output tends to sound robotic - this is inherent to the model architecture
- **MPS Issues:** Has compatibility issues with Apple Silicon MPS - use CPU instead
- **Sample Rate:** Lower output quality at 16kHz (vs 24kHz for others)
- **No Streaming:** Not designed for real-time streaming applications

In [ ]:
# Seamless M4T v2 Basic Usage Example
from transformers import AutoProcessor, SeamlessM4Tv2Model
import torch

# Load model
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large", use_fast=False)
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large")
model.to("cuda")  # or "cpu" for Apple Silicon
model.eval()

# Generate
text = "Hello, thank you for applying to this position."
inputs = processor(text=text, src_lang="eng", return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        tgt_lang="eng",
        speaker_id=1,
        do_sample=True,
        temperature=0.7,
        num_beams=1
    )

# Extract and save audio
audio = output[0].cpu().numpy().squeeze()
import soundfile as sf
sf.write("output.wav", audio, 16000)

## 2.6 Audio Samples - Seamless M4T v2

*Add your audio files here*

In [ ]:
# Placeholder for Seamless audio samples
from IPython.display import Audio, display

# Uncomment and update paths to your audio files
# display(Audio("outputs/seamless/recommended_phrase_1.wav"))
# display(Audio("outputs/seamless/recommended_phrase_2.wav"))

---
# 3. VibeVoice Realtime 0.5B

**Developer:** Microsoft  
**Model:** microsoft/VibeVoice-Realtime-0.5B  
**Architecture:** Qwen2.5-0.5B based with DDPM diffusion

## 3.1 License

- **License:** Research/Non-Commercial
- **Commercial Use:** No - Developers explicitly advise against commercial use

> **Warning:** This model is intended for research purposes only. Do not use in production commercial applications.

## 3.2 Installation

```bash
# Clone and install from source
git clone https://github.com/vibevoice-community/VibeVoice.git
cd VibeVoice
pip install -e .

# Download model weights
huggingface-cli download microsoft/VibeVoice-Realtime-0.5B --local-dir ./checkpoints/0.5B
```

## 3.3 Tunable Parameters

| Parameter | Range | Default | Description |
|-----------|-------|---------|-------------|
| `cfg_scale` | 1.0 - 2.0 | 1.5 | Classifier-free guidance scale (higher = more faithful to text) |
| `ddpm_steps` | 3 - 10 | 5 | Diffusion steps (higher = better quality but slower) |

### Recommended Settings
```python
cfg_scale=1.5, ddpm_steps=5
```

### Quality vs Speed Tradeoff
- **Fast (lower quality):** ddpm_steps=3
- **Balanced:** ddpm_steps=5
- **High quality (slower):** ddpm_steps=10

## 3.4 Special Features

### Ultra-Low Latency Streaming
- Optimized for real-time applications with ~300ms latency
- DDPM-based diffusion for fast inference

### Flash Attention 2 Support
```python
# Automatically uses Flash Attention 2 on CUDA for faster inference
# Falls back to SDPA on MPS/CPU
```

### Compact Model Size
- Based on Qwen2.5-0.5B (smaller footprint)
- ~4GB memory requirement

## 3.5 Known Issues

- **Single Speaker Only:** Only "Carter" (male voice) is available
- **Voice Preset Required:** Requires `en-Carter.pt` preset file
- **English Only:** No multilingual support
- **Non-Commercial:** Explicitly not for commercial use

In [ ]:
# VibeVoice Realtime Basic Usage Example
import torch
import copy
from vibevoice.modular.modeling_vibevoice_streaming_inference import VibeVoiceStreamingForConditionalGenerationInference
from vibevoice.processor.vibevoice_streaming_processor import VibeVoiceStreamingProcessor

# Load model
model_path = "microsoft/VibeVoice-Realtime-0.5B"
processor = VibeVoiceStreamingProcessor.from_pretrained(model_path)
model = VibeVoiceStreamingForConditionalGenerationInference.from_pretrained(
    model_path,
    attn_implementation="flash_attention_2",  # or "sdpa" for MPS/CPU
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
model.set_ddpm_inference_steps(num_steps=5)

# Load voice preset
voice_preset = torch.load("demo/voices/streaming_model/en-Carter.pt", map_location="cuda")

# Generate
text = "Hello, thank you for applying to this position."
inputs = processor.process_input_with_cached_prompt(text, cached_prompt=voice_preset)
inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        cfg_scale=1.5,
        all_prefilled_outputs=copy.deepcopy(voice_preset)
    )

# Save audio
audio = outputs.speech_outputs[0].cpu().numpy()
import soundfile as sf
sf.write("output.wav", audio, 24000)

## 3.6 Audio Samples - VibeVoice

*Add your audio files here*

In [ ]:
# Placeholder for VibeVoice audio samples
from IPython.display import Audio, display

# Uncomment and update paths to your audio files
# display(Audio("outputs/vibevoice/recommended_phrase_1.wav"))
# display(Audio("outputs/vibevoice/recommended_phrase_2.wav"))

---
# 4. Optimization Strategies

## 4.1 GPU Memory Optimization

### Mixed Precision (FP16/BF16)
```python
# Use bfloat16 for CUDA devices
model = model.to(dtype=torch.bfloat16)

# Or use automatic mixed precision
with torch.cuda.amp.autocast():
    output = model.generate(**inputs)
```

### Gradient Checkpointing (for fine-tuning)
```python
model.gradient_checkpointing_enable()
```

### Memory-Efficient Attention
```python
# Flash Attention 2 (CUDA only, requires installation)
pip install flash-attn

# SDPA (built into PyTorch 2.0+)
model = Model.from_pretrained(..., attn_implementation="sdpa")
```

## 4.2 Inference Speed Optimization

### Batch Processing
```python
# Process multiple texts in a batch (where supported)
texts = ["Hello", "World", "Test"]
inputs = processor(texts, padding=True, return_tensors="pt")
outputs = model.generate(**inputs)
```

### Model Compilation (PyTorch 2.0+)
```python
# Compile for faster inference
model = torch.compile(model, mode="reduce-overhead")
```

### ONNX Export
```python
# Export to ONNX for deployment
torch.onnx.export(model, dummy_input, "model.onnx")
```

## 4.3 Platform-Specific Optimizations

### NVIDIA CUDA
- Use Flash Attention 2
- Enable TF32 for Ampere+ GPUs: `torch.backends.cuda.matmul.allow_tf32 = True`
- Use bfloat16 precision

### Apple Silicon (MPS)
- Use float32 (MPS has limited fp16 support)
- Use SDPA attention
- Chatterbox has native MPS support and works great

### AMD GPU (DirectML)
```python
import torch_directml
device = torch_directml.device()
model.to(device)
```

### CPU
- Use Intel OpenVINO for Intel CPUs
- Consider quantization (INT8)

---
# 5. Large-Scale Production Deployment

## 5.1 Scaling Strategies

### Horizontal Scaling
```python
# Deploy multiple model instances behind a load balancer
# Each instance handles a subset of requests

# Example with FastAPI + Uvicorn workers
uvicorn app:app --workers 4
```

### Request Batching
```python
# Collect requests and process in batches
import asyncio
from collections import deque

class BatchProcessor:
    def __init__(self, model, batch_size=8, max_wait=0.1):
        self.model = model
        self.batch_size = batch_size
        self.max_wait = max_wait
        self.queue = deque()
    
    async def process(self, text):
        future = asyncio.Future()
        self.queue.append((text, future))
        
        if len(self.queue) >= self.batch_size:
            await self._process_batch()
        
        return await future
```

### Model Sharding (Multi-GPU)
```python
# Distribute model across multiple GPUs
model = Model.from_pretrained(..., device_map="auto")
```

## 5.2 Caching Strategies

### Audio Caching
```python
import hashlib
import redis

class TTSCache:
    def __init__(self, redis_client):
        self.redis = redis_client
    
    def get_cache_key(self, text, params):
        content = f"{text}_{params}"
        return hashlib.sha256(content.encode()).hexdigest()
    
    def get(self, text, params):
        key = self.get_cache_key(text, params)
        return self.redis.get(key)
    
    def set(self, text, params, audio, ttl=3600):
        key = self.get_cache_key(text, params)
        self.redis.setex(key, ttl, audio)
```

### Common Phrase Pre-generation
```python
# Pre-generate audio for frequently used phrases
COMMON_PHRASES = [
    "Please hold while I transfer your call.",
    "Thank you for your patience.",
    "How can I help you today?",
]

def pregenerate_common_phrases(model, phrases):
    cache = {}
    for phrase in phrases:
        audio = model.generate(phrase)
        cache[phrase] = audio
    return cache
```

## 5.3 Monitoring and Reliability

### Metrics to Track
```python
# Key metrics for TTS systems
metrics = {
    "latency_p50": "50th percentile generation time",
    "latency_p99": "99th percentile generation time",
    "rtf": "Real-time factor (generation_time / audio_duration)",
    "throughput": "Requests per second",
    "gpu_memory": "GPU memory utilization",
    "error_rate": "Failed generation percentage",
}
```

### Health Checks
```python
@app.get("/health")
async def health_check():
    try:
        # Quick inference test
        audio = model.generate("test")
        return {"status": "healthy", "model_loaded": True}
    except Exception as e:
        return {"status": "unhealthy", "error": str(e)}
```

### Graceful Degradation
```python
# Fallback to cached audio or simpler model if primary fails
async def generate_with_fallback(text):
    try:
        return await primary_model.generate(text)
    except Exception:
        # Try cache
        cached = cache.get(text)
        if cached:
            return cached
        # Fallback to simpler model
        return await fallback_model.generate(text)
```

## 5.4 Cost Optimization

### GPU Instance Selection

| Model | Minimum GPU | Recommended GPU | Cloud Instance |
|-------|-------------|-----------------|----------------|
| Chatterbox | 8GB VRAM | RTX 3080/4080 | AWS g4dn.xlarge |
| Seamless | 8GB VRAM | RTX 3080/4080 | AWS g4dn.xlarge |
| VibeVoice | 4GB VRAM | RTX 3060 | AWS g4dn.xlarge |

### Spot/Preemptible Instances
- Use spot instances for non-real-time batch processing
- Save 60-90% on compute costs
- Implement checkpoint/resume for long jobs

### Auto-scaling
```yaml
# Kubernetes HPA example
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
spec:
  minReplicas: 1
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
```

---
# 6. Recommendation Summary

## For Commercial/Enterprise Use
**Use Chatterbox** - MIT license, excellent quality, streaming support

## For Research/Prototyping
**Any model** - All three work well for non-commercial use

## For Multilingual Applications
**Use Seamless M4T v2** - 100+ language support (non-commercial only)

## For Ultra-Low Latency Streaming
**Use VibeVoice or Chatterbox** - Both optimized for real-time

## For Best Audio Quality
**Use Chatterbox** - Most natural sounding, supports expressiveness

---
# 7. Side-by-Side Audio Comparison

*Add your comparative audio samples here to listen to the same phrase from all three models*

In [ ]:
# Side-by-side comparison
from IPython.display import Audio, display, HTML

print("Phrase: 'Hello, thank you for applying to this position.'")
print("="*60)

# Uncomment and update paths
# print("\nChatterbox:")
# display(Audio("outputs/chatterbox/recommended_phrase_1.wav"))

# print("\nSeamless M4T v2:")
# display(Audio("outputs/seamless/recommended_phrase_1.wav"))

# print("\nVibeVoice:")
# display(Audio("outputs/vibevoice/recommended_phrase_1.wav"))